# Segment-wise Mixture-of-Experts for Time Series Forecasting --- How to use

In [ ]:
"""
NeuralForecast datasets for easy access to ETT, Weather, ECL, Traffic datasets.
"""
!pip install datasetsforecast

In [ ]:
import zipfile

with zipfile.ZipFile('segmoe_forecast.zip', 'r') as zf:
    # Extract all contents
    zf.extractall()

In [ ]:
import inspect
import torch
import torch.nn as nn

from segmoe_forecast.model import TSFTransformer

from segmoe_forecast.model.Config import SmallConfig
from segmoe_forecast.data_provider.loaders import get_custom_data_loaders
from segmoe_forecast.utils import CosineLRDecay, EarlyStopping, LoadBalancingLoss, Trainer
from segmoe_forecast.utils.Metrics import eval_forecast_horizons

In [ ]:
device= 'cuda' if torch.cuda.is_available() else 'cpu'

# count how many trainable weights the model has
def count_parameters(model) -> None:
    total_params= sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Number of parameters: {total_params}')

In [ ]:
from segmoe_forecast.data_provider import DataLoaders

dataset_name= 'ETTm1'

df, *_= DataLoaders.load_data(dataset_name)
df.columns

100%|██████████| 314M/314M [00:10<00:00, 30.7MiB/s]


Index(['date', 'HUFL', 'HULL', 'LUFL', 'LULL', 'MUFL', 'MULL', 'OT'], dtype='object')

# Training Backend Setup

In [ ]:
use_fused= False
use_flashAttn= False

if device== 'cuda':
    # TF32 computationally more efficient (slightly the same precision of FP32)
    torch.backends.cudnn.conv.fp32_precision='tf32'
    # enable flash attention
    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cudnn.deterministic= True
    # create AdamW optimizer and use the fused version of it is available
    fused_available= 'fused' in inspect.signature(torch.optim.AdamW).parameters
    # fused is a lot faster when it is available and when running on cuda
    use_fused= fused_available
    use_flashAttn= torch.backends.cuda.flash_sdp_enabled()

print(f"Using fused AdamW: {use_fused}; FlashAttention available: {use_flashAttn}")

Using fused AdamW: True; FlashAttention available: True


# --- Encoder --- Weather (d_model= 128)

In [ ]:
root_path= './'
dataset_name= 'weather'
from_csv= False
btc_size= 256

time_covariates= True
block_size = 512
patch_width= 8
out_width  = 4
n_outputs  = 96

(
    # Train / Val
    train_loader_w, val_loader_w, tds_scaler_w,
    # Test
    test_loader_w_96, test_loader_w_192, test_loader_w_336, test_loader_w_720,

)= get_custom_data_loaders(
    root_path, dataset_name, from_csv, btc_size, time_covariates, patch_width, block_size, out_width
)

print(f"Weather data -- number of batches (train, val, test)")
print(len(train_loader_w), len(val_loader_w), len(test_loader_w_96))

for x, y, covx, covy in train_loader_w:
    print(x.shape, y.shape, covx.shape, covy.shape)
    break


Weather data -- number of batches (train, val, test)
142 21 41
torch.Size([256, 21, 512]) torch.Size([256, 21, 512]) torch.Size([256, 5, 512]) torch.Size([256, 5, 512])


In [ ]:
"""
Training with Weather, block_size=512, patch_width=8, width_factor=4, batch_size=256
"""

channels= 21              #### <------------- multivariate

train_loader= train_loader_w
val_loader  = val_loader_w
test_loader = test_loader_w_96
scaler_obj  = tds_scaler_w

epochs= 20
learning_rate= 3.2e-4
min_lr= 1.2e-5

filename='segmoe_small_weather_checkpoint_p8_4pout_3555_b256'

config= SmallConfig(
    patch_width=patch_width, channels=channels, n_outputs=n_outputs, width_factor=out_width,
    block_size=block_size, n_experts=4, top_k_experts=1, exp_segment_size=[3,5,5,5],
)
model_seg_moe= TSFTransformer.from_config(config).to(device)
count_parameters(model_seg_moe)


# See https://arxiv.org/abs/2402.02592
# train_loader has size X, so Y epochs have X * Y steps
steps= len(train_loader) * epochs

max_lr= learning_rate
warmup_steps= steps * 0.1  # 10% of warmup steps
max_steps= steps

optimizer= model_seg_moe.setup_optimizer(
    learning_rate, weight_decay=1e-1, betas=(0.9, 0.95), verbose=True
)

# for decreasing learning rate -- the CosineLRDecay is designed to be used per step
scheduler= CosineLRDecay(optimizer, min_lr, max_lr, warmup_steps, max_steps)
# terminate training when the validation loss (per epoch) does not improve
early_stopping= EarlyStopping(patience=5, min_delta=0.05)

# See https://arxiv.org/abs/2409.16040
criterion= nn.HuberLoss(reduction='none', delta=2.0)
#criterion= nn.MSELoss(reduction='none')
aux_criterion= LoadBalancingLoss(config.n_experts, config.top_k_experts, alpha=0.02)

trainer= Trainer(
    model_seg_moe, device, train_loader, scaler_obj, val_loader, test_loader,
    criterion, optimizer, scheduler, aux_criterion, early_stopping, time_covariates,
    do_validation=True, checkpointing=True, filename=filename,
    verbose=True
)

Number of parameters: 6917408
Num decayed parameter tensors: 70, with 6914720 parameters
Num non-decayed parameter tensors: 25, with 2688 parameters
Using fused AdamW: True


In [ ]:
trainer.train(epochs, use_bf16=True)

Validating: 100%|██████████| 21/21 [00:03<00:00,  5.91it/s]


Train loss: 45.8467
Valid loss: 42.3326 | dt/epoch: 36259.83ms
======= Epoch 1 Completed =======
[INFO] Checkpoint saved at 'checkpoints/segmoe_small_weather_checkpoint_p8_4pout_3555_b256.pth'


Validating: 100%|██████████| 21/21 [00:03<00:00,  5.93it/s]


Train loss: 16.1289
Valid loss: 18.2861 | dt/epoch: 36228.28ms
======= Epoch 2 Completed =======
[INFO] Checkpoint saved at 'checkpoints/segmoe_small_weather_checkpoint_p8_4pout_3555_b256.pth'


Validating: 100%|██████████| 21/21 [00:03<00:00,  5.96it/s]


...

Train loss: 7.0514
Valid loss: 5.1268 | dt/epoch: 36000.48ms
======= Epoch 18 Completed =======


Validating: 100%|██████████| 21/21 [00:03<00:00,  5.95it/s]


Train loss: 7.0456
Valid loss: 5.1545 | dt/epoch: 36126.96ms
======= Epoch 19 Completed =======


Validating: 100%|██████████| 21/21 [00:03<00:00,  5.95it/s]

Train loss: 7.0387
Valid loss: 5.1503 | dt/epoch: 35992.03ms
======= Epoch 20 Completed =======
Best Validation Loss: 5.0264 (Epoch 8)


In [ ]:
"""
Training with Weather, block_size=512, patch_width=8, width_factor=4, batch_size=256
"""

epoch, best_val_loss= trainer.load_checkpoint('checkpoints/'+filename+'.pth')

avg_mse, avg_mae= eval_forecast_horizons(
    model_seg_moe, trainer, "Weather", test_loader_w_96, test_loader_w_192, test_loader_w_336,
    test_loader_w_720
)

print(f"\nAverage MSE: {avg_mse:.4f}, Average MAE: {avg_mae:.4f}")

[INFO] Checkpoint loaded from 'checkpoints/segmoe_small_weather_checkpoint_p8_4pout_3555_b256.pth'. Resuming training with best validation loss of 5.0264.
Weather
Forecast horizon: 96


Testing: 100%|██████████| 41/41 [00:18<00:00,  2.23it/s]


MSE: 0.14610348641872406
MAE: 0.1880183219909668

Forecast horizon: 192


Testing: 100%|██████████| 41/41 [00:35<00:00,  1.16it/s]


MSE: 0.19034485518932343
MAE: 0.23084595799446106

Forecast horizon: 336


Testing: 100%|██████████| 40/40 [01:03<00:00,  1.58s/it]


MSE: 0.24219335615634918
MAE: 0.27145642042160034

Forecast horizon: 720


Testing: 100%|██████████| 39/39 [02:06<00:00,  3.25s/it]


MSE: 0.31460535526275635
MAE: 0.32371261715888977

Average MSE: 0.2233, Average MAE: 0.2535


# End